# Forex Strategy Lab — Multi-Phase AWP Pipeline

A larger project that chains **4 AWP workflow phases** together:

| Phase | Task | Outputs |
|-------|------|---------|
| **1. Data Fetch & Prep** | Fetch OHLCV data via yfinance, clean, detect anomalies, fill gaps, add datetime index | Cleaned DataFrame |
| **2. Feature Engineering** | Compute 10+ technical indicators (SMA, EMA, RSI, MACD, Bollinger, ATR, etc.) | Enriched DataFrame |
| **3. Strategy Backtest** | Generate buy/sell signals from indicator combos, simulate trades, compute PnL | Trades log + equity curve |
| **4. Report & Visualization** | 10+ publication-quality plots (incl. equity curves), performance metrics, full Markdown report | PNG plots + report.md |

Each phase feeds its outputs into the next phase as inputs.

> **Note:** Libraries like `yfinance` and `matplotlib` are **not** pre-installed — each agent installs them inside the sandbox as needed.

## Setup

In [1]:
import sys, os, json, time
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# AWP source
AWP_SRC = Path.home() / "projects" / "agent-workflow-protocol" / "reference" / "python" / "src"
if str(AWP_SRC) not in sys.path:
    sys.path.insert(0, str(AWP_SRC))

load_dotenv(Path.home() / "projects" / "awp" / ".env")
os.environ["LLM_API_KEY"] = os.environ["OPENROUTER_API_KEY"]

MODEL = "openrouter/" + os.getenv("OPENROUTER_MODEL", "openai/gpt-5-nano")
OUTPUT_BASE = (Path.cwd() / "output_coding_project").resolve()
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

from awp.data import AgentWorkflow

print(f"Model:  {MODEL}")
print(f"Output: {OUTPUT_BASE}")

Model:  openrouter/openai/gpt-5-nano
Output: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_coding_project


In [2]:
# Configuration for yfinance data fetch (agent will install yfinance in sandbox)
TICKER = "BTC-USD"
PERIOD = "1mo"
INTERVAL = "1m"

print(f"Ticker:   {TICKER}")
print(f"Period:   {PERIOD}")
print(f"Interval: {INTERVAL}")

Ticker:   BTC-USD
Period:   1mo
Interval: 1m


In [3]:
# Helper: run a workflow phase and print summary
def run_phase(name, task, inputs, output_subdir, extra_kwargs=None):
    phase_dir = str(OUTPUT_BASE / output_subdir)
    Path(phase_dir).mkdir(parents=True, exist_ok=True)

    kwargs = dict(
        inputs=inputs,
        task=task,
        model=MODEL,
        max_loops=12,
        max_total_tokens=5_000_000,
        max_wall_time=6000,
        max_tool_calls=150,
        max_total_workers=300,
        max_depth=10,
        sandbox="subprocess",
        code_mode=True,
        tool_creation=True,
        verbose=True,
        output_dir=phase_dir,
    )
    if extra_kwargs:
        kwargs.update(extra_kwargs)

    print(f"\n{'='*60}")
    print(f"  PHASE: {name}")
    print(f"  Task:  {task[:80]}...")
    print(f"{'='*60}\n")

    t0 = time.time()
    result = AgentWorkflow(**kwargs).run()
    elapsed = time.time() - t0

    print(f"\n--- {name} complete ---")
    print(f"  Status:    {result['status']}")
    print(f"  Time:      {elapsed:.1f}s")
    print(f"  Workers:   {result['metadata']['workers_spawned']}")
    print(f"  Artifacts: {len(result.get('artifacts', []))} files")
    return result

---
## Phase 1: Data Fetching, Cleaning & Validation

Fetch OHLCV data via yfinance (installed by the agent in the sandbox), then clean and validate it.

In [4]:
phase1 = run_phase(
    name="Data Fetching, Cleaning & Validation",
    task=f"""Fetch and clean OHLCV forex data. Steps:

**Data Fetching (install yfinance first: `pip install yfinance`):**
1. Install yfinance via pip in the sandbox.
2. Use yfinance to download OHLCV data for ticker '{TICKER}', period '{PERIOD}', interval '{INTERVAL}'.
3. Reset the index so 'datetime' is a regular column. Rename columns to lowercase: datetime, open, high, low, close, volume.

**Data Cleaning:**
4. Ensure all price columns (open, high, low, close) are float64, volume is int.
5. Check for and report: missing values, duplicate timestamps, rows where high < low or high < open/close.
6. Forward-fill any missing prices, drop duplicate timestamps (keep last).
7. Add columns: 'spread' = high - low, 'body' = abs(close - open), 'is_bullish' = close > open.
8. Resample to 5-minute bars (OHLCV aggregation: first/max/min/last/sum) and save as '5min_bars'.
9. Save a CSV of the cleaned 1-min data as 'cleaned_1min.csv'.
10. Save a CSV of the 5-min resampled data as 'resampled_5min.csv'.
11. Return a dict with keys: 'cleaning_report' (text summary of issues found),
    'rows_before', 'rows_after', 'rows_5min', 'anomalies_found' (count), and 'ticker' (str).
""",
    inputs={"ticker": TICKER, "period": PERIOD, "interval": INTERVAL},
    output_subdir="phase1_cleaning",
)

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_coding_project/phase1_cleaning
INFO:awp.data.workflow:Starting delegation loop: task=Fetch and clean OHLCV forex data. Steps:

**Data Fetching (install yfinance firs
INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-29_20-34-27_d4d08111] depth=0 starting: Fetch and clean OHLCV forex data. Steps:

**Data Fetching (install yfinance firs
INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===
DEBUG:awp.runtime.llm:LLM request: model=openai/gpt-5-nano, messages=2, tools=0



  PHASE: Data Fetching, Cleaning & Validation
  Task:  Fetch and clean OHLCV forex data. Steps:

**Data Fetching (install yfinance firs...



INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:awp.runtime.delegation_loop_runner:  Spawning worker: yfinance_ohclv_fetcher
INFO:awp.runtime.delegation_loop_runner:Worker yfinance_ohclv_fetcher: temperature=0.20 (from default)
DEBUG:awp.runtime.delegation_loop_runner:Worker yfinance_ohclv_fetcher envelope:
{
  "worker_id": "yfinance_ohclv_fetcher",
  "instructions": "1) Install yfinance in the sandbox environment. 2) Use yfinance to download OHLCV data for ticker BTC-USD with period=1mo and interval=1m. 3) Reset the index so 'datetime' is a regular column and rename columns to lowercase: datetime, open, high, low, close, volume. 4) Cast price columns to float64 and volume to int. 5) Gather a cleaning report: count missing values, count duplicate timestamps, and count anomalies where high < low or high < open/close. 6) Forward-fill missing prices and drop duplicate timestamps (keeping the last). 7) Add columns: spread = high - low, bod


  AWP DELEGATION LOOP DEBUG REPORT
  Model:         openrouter/openai/gpt-5-nano
  Worker model:  openrouter/openai/gpt-5-nano
  Budget:        loops=12, workers=300, tokens=5,000,000, wall_time=6000s, depth=10

  ────────────────────────────────────────────────────────
  Iteration 001
  ────────────────────────────────────────────────────────
    ────────────────────────────────────────────────
    MANAGER DECISION
    ────────────────────────────────────────────────
    Decision:    delegate
    Reasoning:
      | The task involves data fetching, cleaning, and resampling which is best handled by a dedicated worker using Python (code.execute). A single well-defined worker can fetch data via yfinance, perform all cleaning steps, create 1-min and 5-min outputs, and return a structured report.
    Full Manager Decision JSON:
      {
        "decision": "delegate",
        "reasoning": "The task involves data fetching, cleaning, and resampling which is best handled by a dedicated worker 

In [5]:
# Load cleaned data from Phase 1 for next phases
# Files are saved inside output/<run_id>/ subdirectory
phase1_out = sorted((OUTPUT_BASE / "phase1_cleaning" / "output").glob("*/cleaned_1min.csv"))[-1].parent
cleaned_1min = pd.read_csv(phase1_out / "cleaned_1min.csv", parse_dates=["datetime"])
resampled_5min = pd.read_csv(phase1_out / "resampled_5min.csv", parse_dates=["datetime"])
print(f"Cleaned 1min: {cleaned_1min.shape}  |  Resampled 5min: {resampled_5min.shape}")

IndexError: list index out of range

---
## Phase 2: Technical Indicator Engineering

Compute a full suite of technical indicators on the 5-minute data. These will be used as features for the strategy signals.

In [ ]:
phase2 = run_phase(
    name="Technical Indicator Engineering",
    task="""Compute technical indicators on the 5-minute OHLCV data in 'ohlcv_5min'.
Add ALL of the following as new columns, using standard formulas:

1.  SMA_20, SMA_50 — Simple Moving Averages (20 and 50 periods)
2.  EMA_12, EMA_26 — Exponential Moving Averages
3.  MACD, MACD_signal, MACD_hist — MACD (12,26,9)
4.  RSI_14 — Relative Strength Index (14 periods)
5.  BB_upper, BB_mid, BB_lower — Bollinger Bands (20, 2 std)
6.  ATR_14 — Average True Range (14 periods)
7.  STOCH_K, STOCH_D — Stochastic Oscillator (14,3,3)
8.  OBV — On Balance Volume
9.  VWAP — Volume Weighted Average Price (session-based, reset daily)
10. ADX_14 — Average Directional Index (14 periods)
11. CCI_20 — Commodity Channel Index (20 periods)
12. WILLR_14 — Williams %R (14 periods)
13. MOM_10 — Momentum (10 periods)
14. ROC_10 — Rate of Change (10 periods)

Save the enriched DataFrame as 'enriched_5min.csv'.
Return a dict with: 'indicators_added' (list of column names added),
'total_columns' (int), 'sample_row' (dict of one row with all indicators).
Do NOT use any external packages like ta-lib — implement all formulas in pure pandas/numpy.
""",
    inputs={"ohlcv_5min": resampled_5min},
    output_subdir="phase2_indicators",
)
phase2["result"]

In [ ]:
# Load enriched data for Phase 3
phase2_out = sorted((OUTPUT_BASE / "phase2_indicators" / "output").glob("*/enriched_5min.csv"))[-1].parent
enriched_5min = pd.read_csv(phase2_out / "enriched_5min.csv", parse_dates=["datetime"])
print(f"Enriched 5min: {enriched_5min.shape}")
print(f"Columns: {list(enriched_5min.columns)}")

---
## Phase 3: Strategy Backtesting

Run 3 different trading strategies on the enriched data, simulate trades, and compute performance metrics.

In [ ]:
phase3 = run_phase(
    name="Strategy Backtesting",
    task="""Backtest 3 trading strategies on the enriched 5-minute EURJPY data in 'enriched'.
Initial capital: 10,000 USD. Position size: 1 lot per trade. No leverage modeling needed.

**Strategy A — SMA Crossover:**
- BUY when SMA_20 crosses above SMA_50, SELL when SMA_20 crosses below SMA_50.
- Stop-loss: 1.5x ATR_14. Take-profit: 3x ATR_14.

**Strategy B — RSI + Bollinger Bands:**
- BUY when RSI_14 < 30 AND close < BB_lower. SELL when RSI_14 > 70 AND close > BB_upper.
- Stop-loss: 2x ATR_14. Take-profit: 2x ATR_14.

**Strategy C — MACD + Stochastic:**
- BUY when MACD_hist crosses above 0 AND STOCH_K < 20. SELL when MACD_hist crosses below 0 AND STOCH_K > 80.
- Stop-loss: 1.5x ATR_14. Take-profit: 2.5x ATR_14.

For each strategy, compute:
- Total trades, win rate, average win/loss, profit factor
- Max drawdown (%), Sharpe ratio (annualized, assume 252*24*12 5-min bars/year)
- Equity curve (cumulative PnL over time)

Save:
- 'trades_A.csv', 'trades_B.csv', 'trades_C.csv' — each row: entry_time, exit_time, direction, entry_price, exit_price, pnl, exit_reason
- 'equity_curves.csv' — columns: datetime, equity_A, equity_B, equity_C
- 'strategy_comparison.csv' — one row per strategy with all performance metrics

Return a dict with keys: 'strategy_A', 'strategy_B', 'strategy_C' (each containing the metrics dict),
and 'best_strategy' (name of the one with highest Sharpe ratio).
""",
    inputs={"enriched": enriched_5min},
    output_subdir="phase3_backtest",
)
phase3["result"]

---
## Phase 4: Report & Visualization

Generate 10+ publication-quality plots and a comprehensive Markdown report summarizing all findings.

In [ ]:
# Gather all phase outputs for the final report
phase3_out = sorted((OUTPUT_BASE / "phase3_backtest" / "output").glob("*/equity_curves.csv"))[-1].parent
equity_curves = pd.read_csv(phase3_out / "equity_curves.csv", parse_dates=["datetime"])
strategy_comp = pd.read_csv(phase3_out / "strategy_comparison.csv")
trades_a = pd.read_csv(phase3_out / "trades_A.csv")
trades_b = pd.read_csv(phase3_out / "trades_B.csv")
trades_c = pd.read_csv(phase3_out / "trades_C.csv")

print(f"Equity curves: {equity_curves.shape}")
print(f"Strategy comparison:\n{strategy_comp.to_string()}")
print(f"Trades: A={len(trades_a)}, B={len(trades_b)}, C={len(trades_c)}")

In [ ]:
phase4 = run_phase(
    name="Report & Visualization",
    task="""Create a comprehensive analysis report with 10+ plots for the EURJPY strategy backtest.

**IMPORTANT: First install matplotlib in the sandbox: `pip install matplotlib`**

You have these inputs:
- 'enriched_5min': 5-min OHLCV data with all technical indicators
- 'equity_curves': equity curves for strategies A, B, C (columns: datetime, equity_A, equity_B, equity_C)
- 'strategy_comparison': performance metrics table (one row per strategy)
- 'trades_A', 'trades_B', 'trades_C': individual trade logs
- 'phase1_report': text summary from data cleaning
- 'phase3_result': backtest result dict with metrics per strategy

Create these plots (save each as PNG, 1200x800px, dpi=150, tight_layout):
1.  price_overview.png — EURJPY close price with SMA_20/SMA_50 overlay (full time range)
2.  candlestick_sample.png — 2-day candlestick chart with Bollinger Bands
3.  indicator_dashboard.png — 4-panel subplot: price, RSI, MACD histogram, Stochastic
4.  equity_comparison.png — **ALL 3 equity curves overlaid**, with drawdown shading, legend, and grid. This is a key deliverable.
5.  drawdown_chart.png — drawdown (%) over time for all 3 strategies
6.  trade_distribution.png — histogram of PnL per trade for each strategy (side by side)
7.  win_loss_bars.png — grouped bar chart: wins vs losses count per strategy
8.  monthly_returns.png — heatmap of monthly returns for the best strategy
9.  volume_profile.png — volume distribution by hour of day (bar chart)
10. correlation_matrix.png — heatmap of correlations between all indicators
11. strategy_scatter.png — scatter plot: Sharpe ratio vs max drawdown for 3 strategies
12. cumulative_trades.png — cumulative number of trades over time for each strategy

Also create 'report.md' — a full Markdown report with:
- Executive summary (best strategy, key metrics)
- Data overview (rows, date range, cleaning issues)
- Technical indicators computed
- Strategy descriptions and rules
- Performance comparison table
- Key findings and recommendations
- References to all plot filenames

Return a dict with 'plots' (list of filenames), 'report_path', and 'summary' (3-sentence executive summary).
""",
    inputs={
        "enriched_5min": enriched_5min,
        "equity_curves": equity_curves,
        "strategy_comparison": strategy_comp,
        "trades_A": trades_a,
        "trades_B": trades_b,
        "trades_C": trades_c,
        "phase1_report": str(phase1.get("result", {}).get("cleaning_report", "N/A")),
        "phase3_result": json.dumps(phase3.get("result", {}), default=str),
    },
    output_subdir="phase4_report",
)
phase4["result"]

---
## Pipeline Summary

In [ ]:
# Final overview of all outputs
print("=" * 60)
print("  FOREX STRATEGY LAB — ALL OUTPUTS")
print("=" * 60)

for phase_dir in sorted(OUTPUT_BASE.iterdir()):
    if phase_dir.is_dir():
        files = list(phase_dir.rglob("*"))
        files = [f for f in files if f.is_file()]
        total_size = sum(f.stat().st_size for f in files)
        print(f"\n{phase_dir.name}/  ({len(files)} files, {total_size/1024:.1f} KB)")
        for f in sorted(files):
            print(f"  {f.relative_to(phase_dir)}  ({f.stat().st_size:,} bytes)")

In [ ]:
# Display the final report
phase4_out = OUTPUT_BASE / "phase4_report" / "output"
report_candidates = sorted(phase4_out.glob("*/report.md")) if phase4_out.exists() else []
if report_candidates:
    from IPython.display import Markdown, display
    display(Markdown(report_candidates[-1].read_text()))
else:
    print("Report not found — check phase 4 output.")

---
## Equity Curves

Display the equity curves generated by Phase 4 directly in the notebook.

In [ ]:
# Display equity curve plot from Phase 4 output
from IPython.display import Image, display

phase4_out = OUTPUT_BASE / "phase4_report" / "output"
equity_plots = sorted(phase4_out.glob("*/equity_comparison.png")) if phase4_out.exists() else []

if equity_plots:
    display(Image(filename=str(equity_plots[-1]), width=1000))
    print(f"Source: {equity_plots[-1]}")
else:
    print("Equity curve plot not found — check Phase 4 output.")